<p align="center">
  <img src="https://a2.espncdn.com/combiner/i?img=%2Fphoto%2F2024%2F1218%2Fr1429402_1296x729_16%2D9.jpg" width="70%">
</p>

<div id="inicio"></div>
<a id="indice"></a>
<div style="background-color: #1a1a1a; padding: 30px; border-radius: 15px; border: 1px solid #2e7d32; text-align: center;">
    <h1 style="color: #ffffff; font-family: 'Segoe UI', sans-serif; margin-bottom: 5px;">FINAL: MUNDIAL 2022</h1>
    <p style="color: #a5d6a7; font-size: 18px; margin-top: 0;">Análisis de Datos con StatsBomb & mplsoccer</p>
    <hr style="border: 0.5px solid #00796b; width: 80%;">
    <div style="text-align: left; display: inline-block; color: white;">
        <ul style="list-style-type: none; line-height: 2;">
            <li><a href="#seccion1" style="color: #00796b; text-decoration: none;"><b>01.</b> Obtención de Información</a></li>
            <li><a href="#seccion2" style="color: #00796b; text-decoration: none;"><b>02.</b> Estadísticas del Partido</a></li>
            <li><a href="#seccion3" style="color: #00796b; text-decoration: none;"><b>03.</b> Mapas de Equipos</a></li>
            <li><a href="#seccion4" style="color: #00796b; text-decoration: none;"><b>04.</b> Análisis Individual de Jugador</a></li>
            <li><a href="#seccion5" style="color: #00796b; text-decoration: none;"><b>05.</b> Dashboard del Partido</a></li>
            <li><a href="#seccion6" style="color: #00796b; text-decoration: none;"><b>06.</b> Análisis Completo del Mundial</a></li>
            <li><a href="#extras" style="color: #ffd54f; text-decoration: none;"><b>EXTRAS.</b></a></li>
        </ul>
    </div>
</div>

![Pandas](https://img.shields.io/badge/Pandas-150458?style=for-the-badge&logo=pandas&logoColor=white)
![NumPy](https://img.shields.io/badge/numpy-%23013243.svg?style=for-the-badge&logo=numpy&logoColor=white)
![Matplotlib](https://img.shields.io/badge/Matplotlib-%23ffffff.svg?style=for-the-badge&logo=Matplotlib&logoColor=black)

<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

<a id='seccion1'></a>
<div style="padding: 10px; border-bottom: 2px solid #00796b;">
    <h2 style="color: #00796b;">1. Obtención de información del partido</h2>
</div>

> **Objetivo:** Conexión a la API de StatsBomb y carga de eventos para el partido seleccionado.
[Volver al índice](#inicio)

In [ ]:
from statsbombpy import sb
import pandas as pd

# Listar partidos del Mundial 2022
partidos = sb.matches(competition_id=43, season_id=106)

# Filtrar el match_id de la final (Argentina vs Francia)
match_id = 3869685

# Descargar eventos del partido
eventos = sb.events(match_id=match_id)

# Vistazo rápido a las columnas para saber con qué trabajamos
print(eventos.columns)

<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

<a id='seccion2'></a>
<div style="padding: 10px; border-bottom: 2px solid #00796b;">
    <h2 style="color: #00796b;">2. Análisis de estadísticas del partido</h2>
</div>

Calcularemos las métricas clave para entender el desarrollo del encuentro:
* **xG Total** por equipo.
* **Máximo pasador** y **Recuperador** del encuentro.
* Análisis de **ocupación por tercios** del campo.

[Volver al índice](#inicio)

In [ ]:
# Equipo con más xG
xg_por_equipo = eventos.groupby('team')['shot_statsbomb_xg'].sum().reset_index()
equipo_mas_xg = xg_por_equipo.sort_values(by='shot_statsbomb_xg', ascending=False).iloc[0]

print(f"El equipo con más xG fue {equipo_mas_xg['team']} con {equipo_mas_xg['shot_statsbomb_xg']:.2f}")

In [ ]:
# Jugador con más pases
pases = eventos[eventos.type == 'Pass']
top_pasador = pases['player'].value_counts().idxmax()
total_pases = pases['player'].value_counts().max()

# Jugador con más recuperaciones
recuperaciones = eventos[eventos.type == 'Ball Recovery']
top_recuperador = recuperaciones['player'].value_counts().idxmax()
total_recuperaciones = recuperaciones['player'].value_counts().max()

print(f"Líder de pases: {top_pasador} ({total_pases})")
print(f"Líder de recuperaciones: {top_recuperador} ({total_recuperaciones})")

<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

In [ ]:
# Eventos con coordenadas
df_coords = eventos.dropna(subset=['location']).copy()

# Coordenada x
df_coords['x'] = df_coords['location'].apply(lambda loc: loc[0])

# Definimos tercios
df_coords['tercio'] = pd.cut(
    df_coords['x'],
    bins=[0, 40, 80, 120],
    labels=['Defensivo', 'Medio', 'Ofensivo'],
    include_lowest=True
)

# Conteo
acciones_tercio = df_coords['tercio'].value_counts()

zona_mas_activa = acciones_tercio.idxmax()
conteo_zona = acciones_tercio.max()

print(
    f"La zona con más acción fue el tercio {zona_mas_activa} "
    f"con {conteo_zona} eventos."
)

acciones_tercio

<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

<a id='seccion3'></a>
<div style="padding: 10px; border-bottom: 2px solid #00796b;">
    <h2 style="color: #00796b;">3. Mapas de equipos</h2>
</div>

Visualización espacial de los eventos de ambos equipos:
1.  **Mapa de Tiros:** 
2.  **Mapa de Pases:** 

<a id='seccion5'></a>
<div style="padding: 10px; border-bottom: 2px solid #00796b;">
    <h2 style="color: #00796b;">5. Dashboard del Partido</h2>
</div>

<div class="alert alert-info">
    Resumen visual integrado que combina las métricas de equipos, jugadores destacados y el resultado final en una sola pieza gráfica.
</div>

[Volver al índice](#indice)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Limpieza de coordenadas
df = eventos.copy()
mask_loc = df['location'].notna()
coords = pd.DataFrame(df.loc[mask_loc, 'location'].tolist(), index=df.loc[mask_loc].index)
df.loc[mask_loc, ['x', 'y']] = coords.values

mask_pass = df['pass_end_location'].notna()
pass_coords = pd.DataFrame(df.loc[mask_pass, 'pass_end_location'].tolist(), index=df.loc[mask_pass].index)
df.loc[mask_pass, ['end_x', 'end_y']] = pass_coords.values

# Equipos
df_arg = df[df.team == 'Argentina']
df_fra = df[df.team == 'France']

# Jugadores destacados
destacados = (
    df.groupby('player')
    .agg(
        pases=('type', lambda x: (x == 'Pass').sum()),
        recuperaciones=('type', lambda x: (x == 'Ball Recovery').sum())
    )
    .sort_values(by=['pases', 'recuperaciones'], ascending=False)
    .head(3)
)
texto_dest = "\n".join([f"{i+1}. {p} – Pases: {r.pases}, Recup: {r.recuperaciones}" for i, (p, r) in enumerate(destacados.iterrows())])

# Creación del dashboard
fig = plt.figure(figsize=(22, 14), facecolor='#1e1e1e')
gs = fig.add_gridspec(2, 3, width_ratios=[1, 0.8, 1])
pitch = Pitch(pitch_type='statsbomb', pitch_color='grass', line_color='white')

# Tiros Argentina
ax1 = fig.add_subplot(gs[0, 0])
pitch.draw(ax=ax1)
ax1.set_title("Tiros promedio – Argentina", color='white', fontsize=16)
shots_arg = df_arg[(df_arg.type == 'Shot') & (df_arg.x.notna())].copy()
shots_arg['x_zone'] = pd.cut(shots_arg['x'], bins=np.linspace(0, 120, 7))
shots_arg['y_zone'] = pd.cut(shots_arg['y'], bins=np.linspace(0, 80, 5))
shots_avg_arg = shots_arg.groupby(['x_zone', 'y_zone']).agg(x_mean=('x', 'mean'), y_mean=('y', 'mean'), shots=('x', 'count')).reset_index()
# Usamos .values para evitar errores
pitch.scatter(shots_avg_arg.x_mean.values, shots_avg_arg.y_mean.values, s=shots_avg_arg.shots.values * 150, color='#74ACDF', edgecolors='white', alpha=0.8, ax=ax1)

# Tiros Francia
ax2 = fig.add_subplot(gs[0, 2])
pitch.draw(ax=ax2)
ax2.set_title("Tiros promedio – Francia", color='white', fontsize=16)
shots_fra = df_fra[(df_fra.type == 'Shot') & (df_fra.x.notna())].copy()
shots_fra['x_zone'] = pd.cut(shots_fra['x'], bins=np.linspace(0, 120, 7))
shots_fra['y_zone'] = pd.cut(shots_fra['y'], bins=np.linspace(0, 80, 5))
shots_avg_fra = shots_fra.groupby(['x_zone', 'y_zone']).agg(x_mean=('x', 'mean'), y_mean=('y', 'mean'), shots=('x', 'count')).reset_index()
# Usamos .values para evitar errores
pitch.scatter(shots_avg_fra.x_mean.values, shots_avg_fra.y_mean.values, s=shots_avg_fra.shots.values * 150, color='#002395', edgecolors='white', alpha=0.8, ax=ax2)

# Pases Argentina
ax3 = fig.add_subplot(gs[1, 0])
pitch.draw(ax=ax3)
ax3.set_title("Pases por Zona – Argentina", color='white', fontsize=16)
passes_arg = df_arg[(df_arg.type == 'Pass') & (df_arg.pass_outcome.isna()) & (df_arg.x.notna()) & (df_arg.end_x.notna())].copy()
passes_arg['x_zone'] = pd.cut(passes_arg['x'], bins=np.linspace(0, 120, 7))
passes_arg['y_zone'] = pd.cut(passes_arg['y'], bins=np.linspace(0, 80, 5))
passes_avg_arg = passes_arg.groupby(['x_zone', 'y_zone']).agg(x_mean=('x', 'mean'), y_mean=('y', 'mean'), end_x_mean=('end_x', 'mean'), end_y_mean=('end_y', 'mean'), count=('x', 'count')).reset_index()
pitch.arrows(passes_avg_arg.x_mean.values, passes_avg_arg.y_mean.values, passes_avg_arg.end_x_mean.values, passes_avg_arg.end_y_mean.values, width=4, color='#74ACDF', alpha=0.7, ax=ax3)

# Pases Francia
ax4 = fig.add_subplot(gs[1, 2])
pitch.draw(ax=ax4)
ax4.set_title("Pases – Francia", color='white', fontsize=16)
passes_fra = df_fra[(df_fra.type == 'Pass') & (df_fra.pass_outcome.isna()) & (df_fra.x.notna()) & (df_fra.end_x.notna())].copy()
passes_fra['x_zone'] = pd.cut(passes_fra['x'], bins=np.linspace(0, 120, 7))
passes_fra['y_zone'] = pd.cut(passes_fra['y'], bins=np.linspace(0, 80, 5))
passes_avg_fra = passes_fra.groupby(['x_zone', 'y_zone']).agg(x_mean=('x', 'mean'), y_mean=('y', 'mean'), end_x_mean=('end_x', 'mean'), end_y_mean=('end_y', 'mean'), count=('x', 'count')).reset_index()
pitch.arrows(passes_avg_fra.x_mean.values, passes_avg_fra.y_mean.values, passes_avg_fra.end_x_mean.values, passes_avg_fra.end_y_mean.values, width=4, color='#002395', alpha=0.7, ax=ax4)



# Texto central
ax_center = fig.add_subplot(gs[:, 1])
ax_center.axis('off')
ax_center.text(0.5, 0.75, "Argentina 3 – 3 Francia\n(Penales 4–2)", ha='center', va='center', fontsize=26, color='white', weight='bold')
ax_center.text(0.5, 0.3, f"Jugadores Destacados:\n\n{texto_dest}", ha='center', va='center', fontsize=14, color='#f1c40f')

plt.suptitle("Dashboard Final: Argentina vs Francia", fontsize=30, color='white', y=0.98)
plt.show()

<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

<a id='seccion4'></a>
<div style="padding: 10px; border-bottom: 2px solid #00796b;">
    <h2 style="color: #00796b;">4. Análisis Individual</h2>
</div>

Foco detallado en el rendimiento de un jugador clave mediante:
* **Mapa de calor** (Zonas de mayor influencia).
* **Mapa de Tiros** (Finalización).
* **Mapas de Acción** (Pases, recuperaciones, intercepciones y faltas).

[Volver al índice](#indice)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Buscar eventos de Mac Allister

jugador = 'Alexis Mac Allister'
df_alexis = eventos[eventos['player'] == jugador].copy()

# Coordenadas iniciales de pases
mask_loc = df_alexis['location'].notna()
coords = pd.DataFrame(
    df_alexis.loc[mask_loc, 'location'].tolist(),
    index=df_alexis.loc[mask_loc].index,
    columns=['x', 'y']
)
df_alexis.loc[mask_loc, ['x', 'y']] = coords

# Coordenadas finales de pases
mask_pass = df_alexis['pass_end_location'].notna()
pass_coords = pd.DataFrame(
    df_alexis.loc[mask_pass, 'pass_end_location'].tolist(),
    index=df_alexis.loc[mask_pass].index,
    columns=['pass_end_x', 'pass_end_y']
)
df_alexis.loc[mask_pass, ['pass_end_x', 'pass_end_y']] = pass_coords


# Graficos

fig, ax = plt.subplot_mosaic([['heat', 'actions']], figsize=(20, 10))
fig.set_facecolor('#1e1e1e')

pitch = Pitch(
    pitch_type='statsbomb',
    pitch_color='grass',
    line_color='#ecf0f1'
)


# Mapa de calor

pitch.draw(ax=ax['heat'])

pitch.kdeplot(
    df_alexis['x'],
    df_alexis['y'],
    ax=ax['heat'],
    fill=True,
    levels=100,
    thresh=0,
    cmap='hot',
    alpha=0.55
)

ax['heat'].set_title(
    f"Mapa de Calor – {jugador}",
    color='white',
    fontsize=18,
    pad=12
)


# Mapa de acciones

pitch.draw(ax=ax['actions'])

# Pases completados
pases_ok = df_alexis[
    (df_alexis['type'] == 'Pass') &
    (df_alexis['pass_outcome'].isna())
]

pitch.arrows(
    pases_ok['x'], pases_ok['y'],
    pases_ok['pass_end_x'], pases_ok['pass_end_y'],
    width=2,
    headwidth=3,
    color='#74ACDF',
    alpha=0.6,
    ax=ax['actions'],
    label='Pases Completados'
)

# Recuperaciones
recuperaciones = df_alexis[df_alexis['type'] == 'Ball Recovery']

pitch.scatter(
    recuperaciones['x'],
    recuperaciones['y'],
    color='#2ECC71',
    s=110,
    edgecolors='black',
    ax=ax['actions'],
    label='Recuperaciones'
)

# Intercepciones
intercepciones = df_alexis[df_alexis['type'] == 'Interception']

pitch.scatter(
    intercepciones['x'],
    intercepciones['y'],
    color='#9B59B6',
    s=110,
    edgecolors='black',
    ax=ax['actions'],
    label='Intercepciones'
)

# Tiros
tiros = df_alexis[df_alexis['type'] == 'Shot'].copy()

# Tamaño de los tiros
if 'shot_statsbomb_xg' in tiros.columns:
    tiros['size'] = tiros['shot_statsbomb_xg'].fillna(0.05) * 1200
else:
    tiros['size'] = 200

pitch.scatter(
    tiros['x'],
    tiros['y'],
    s=tiros['size'],
    marker='*',
    color='#F1C40F',
    edgecolors='white',
    linewidth=1.5,
    ax=ax['actions'],
    label='Tiros (tamaño = xG)'
)

# Faltas cometidas
faltas = df_alexis[df_alexis['type'] == 'Foul Committed']

pitch.scatter(
    faltas['x'],
    faltas['y'],
    marker='x',
    s=160,
    c='white',
    edgecolors='#FF0000',
    linewidth=2,
    ax=ax['actions'],
    label='Faltas Cometidas'
)

 
# Mapa de acciones

ax['actions'].set_title(
    f"Mapa de Acciones – {jugador}",
    color='white',
    fontsize=18,
    pad=12
)

ax['actions'].legend(
    loc='lower center',
    bbox_to_anchor=(0.5, -0.18),
    ncol=3,
    fontsize=12,
    frameon=False
)

plt.show()


# Resumen de acciones
 
print(f"Acciones de {jugador}:")
print(f"• Pases completados: {len(pases_ok)}")
print(f"• Recuperaciones: {len(recuperaciones)}")
print(f"• Intercepciones: {len(intercepciones)}")
print(f"• Tiros: {len(tiros)}")
print(f"• Faltas cometidas: {len(faltas)}")


<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

In [ ]:
from statsbombpy import sb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch, VerticalPitch, Radar

# Desactivar advertencias
import warnings
warnings.filterwarnings('ignore')

# Descargar lista de partidos
print("Descargando lista de partidos...")
partidos = sb.matches(competition_id=43, season_id=106)
match_ids = partidos.match_id.unique()

# Descargar eventos
print(f"Descargando eventos de {len(match_ids)} partidos... (Esto puede tardar unos minutos)")
eventos_list = []
cols_to_keep = ['match_id', 'team', 'player', 'type', 'minute', 'location', 
                'pass_end_location', 'shot_statsbomb_xg', 'shot_outcome', 'pass_outcome']

for mid in match_ids:
    try:
        ev = sb.events(match_id=mid)
        # Filtrar columnas
        cols_existentes = [c for c in cols_to_keep if c in ev.columns]
        eventos_list.append(ev[cols_existentes])
    except:
        pass

df_mundial = pd.concat(eventos_list, ignore_index=True)

# Normalizar coordenadas
df_mundial['x'] = df_mundial['location'].apply(lambda x: x[0] if isinstance(x, list) else np.nan)
df_mundial['y'] = df_mundial['location'].apply(lambda x: x[1] if isinstance(x, list) else np.nan)
df_mundial['end_x'] = df_mundial['pass_end_location'].apply(lambda x: x[0] if isinstance(x, list) else np.nan)

# Crear columna is_goal
# Si existe la columna shot_outcome, usarla para crear is_goal
if 'shot_outcome' in df_mundial.columns:
    df_mundial['is_goal'] = df_mundial['shot_outcome'] == 'Goal'
else:
    df_mundial['is_goal'] = False # Si no existe, asumimos que no hay goles

print("¡Datos del Mundial cargados y procesados!")

<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

<a id='seccion6'></a>
<div style="padding: 10px; border-bottom: 2px solid #00796b;">
    <h2 style="color: #00796b;">6. Análisis Global del Mundial 2022</h2>
</div>

Procesamiento de los **64 partidos** del torneo:
* **Rankings:** Goleadores, pasadores y recuperadores.
* **Scatters:** Correlación entre volumen de tiros/pases y efectividad.
* **Player Report:** Dashboard de 3 canchas para un jugador destacado del torneo.

[Volver al índice](#indice)

In [ ]:
# A. Jugador con más xG en Holanda-Ecuador
match_ned_ecu = partidos[(partidos.home_team == 'Netherlands') & (partidos.away_team == 'Ecuador')].match_id.values[0]
df_ne = df_mundial[df_mundial.match_id == match_ned_ecu]
top_xg = df_ne.groupby('player')['shot_statsbomb_xg'].sum().idxmax()
val_xg = df_ne.groupby('player')['shot_statsbomb_xg'].sum().max()
print(f"1. Jugador con más xG (Holanda-Ecuador): {top_xg} ({val_xg:.2f})")

# B. Pases en Inglaterra vs Irán
match_eng_irn = partidos[(partidos.home_team == 'England') & (partidos.away_team == 'Iran')].match_id.values[0]
df_ei = df_mundial[df_mundial.match_id == match_eng_irn]
# Pases intentados
pases_totales = df_ei[df_ei.type == 'Pass'].groupby('player').size()
# Pases completados (pass_outcome es NaN si es completo)
pases_completos = df_ei[(df_ei.type == 'Pass') & (df_ei.pass_outcome.isna())].groupby('player').size()

print(f"2. Inglaterra-Irán: Más intentos: {pases_totales.idxmax()} ({pases_totales.max()}) | Más completados: {pases_completos.idxmax()} ({pases_completos.max()})")

# C. Pases en Marruecos vs Francia
match_mar_fra = partidos[((partidos.home_team == 'Morocco') | (partidos.home_team == 'France')) & 
                         ((partidos.away_team == 'Morocco') | (partidos.away_team == 'France'))].match_id.values[0]
df_mf = df_mundial[df_mundial.match_id == match_mar_fra]
total_intentos = len(df_mf[df_mf.type == 'Pass'])
total_completos = len(df_mf[(df_mf.type == 'Pass') & (df_mf.pass_outcome.isna())])

print(f"3. Marruecos-Francia: Pases Intentados: {total_intentos} | Completados: {total_completos}")

<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import VerticalPitch

# Filtrar Lionel Messi
df_messi = eventos[
    eventos.player == 'Lionel Andrés Messi Cuccittini'
].copy()

df_messi['x'] = df_messi['location'].apply(
    lambda x: x[0] if isinstance(x, list) else np.nan
)
df_messi['y'] = df_messi['location'].apply(
    lambda x: x[1] if isinstance(x, list) else np.nan
)

df_messi = df_messi[df_messi.x.notna() & df_messi.y.notna()]

# Crear gráfico

fig, axs = plt.subplot_mosaic(
    [['heat', 'pass', 'shot']],
    figsize=(20, 6),
    facecolor='#1e1e1e'
)

vp = VerticalPitch(
    pitch_type='statsbomb',
    pitch_color='#22312b',
    line_color='white'
)

vp.draw(ax=axs['heat'])

bin_stat = vp.bin_statistic(
    df_messi.x,
    df_messi.y,
    statistic='count',
    bins=(12, 8),
    normalize=True
)

# Normalizar
bin_stat['statistic'] = bin_stat['statistic'] / bin_stat['statistic'].max()

vp.heatmap(
    bin_stat,
    ax=axs['heat'],
    cmap='Reds',
    edgecolors='#22312b',
    linewidth=0.4,
    alpha=0.85
)

axs['heat'].set_title(
    "Zonas de Influencia (frecuencia de acciones)",
    color='white',
    fontsize=15
)

# Pases
vp.draw(ax=axs['pass'])

df_messi['end_x'] = df_messi['pass_end_location'].apply(
    lambda x: x[0] if isinstance(x, list) else np.nan
)
df_messi['end_y'] = df_messi['pass_end_location'].apply(
    lambda x: x[1] if isinstance(x, list) else np.nan
)

passes_messi = df_messi[
    (df_messi.type == 'Pass') &
    (df_messi.pass_outcome.isna()) &
    (df_messi.end_x.notna())
].copy()

passes_messi['x_zone'] = pd.cut(passes_messi['x'], bins=np.linspace(0, 120, 9))
passes_messi['y_zone'] = pd.cut(passes_messi['y'], bins=np.linspace(0, 80, 7))

# Promedio de pases
passes_avg = (
    passes_messi
    .groupby(['x_zone', 'y_zone'])
    .agg(
        x_mean=('x', 'mean'),
        y_mean=('y', 'mean'),
        end_x_mean=('end_x', 'mean'),
        end_y_mean=('end_y', 'mean'),
        count=('x', 'count')
    )
    .reset_index()
)

passes_avg = passes_avg[passes_avg['count'] >= 4]
max_count = passes_avg['count'].max()

for _, row in passes_avg.iterrows():
    width = 2 + (row['count'] / max_count) * 5

    vp.arrows(
        row['x_mean'],
        row['y_mean'],
        row['end_x_mean'],
        row['end_y_mean'],
        ax=axs['pass'],
        width=width,
        color='#74ACDF',
        alpha=0.85
    )

# Puntos de tiros
axs['pass'].set_title(
    "Dirección y volumen de pases",
    color='white',
    fontsize=15
)

vp.draw(ax=axs['shot'])

shots = df_messi[df_messi.type == 'Shot']

vp.scatter(
    shots.x,
    shots.y,
    ax=axs['shot'],
    s=shots.shot_statsbomb_xg * 1200,
    c='#ff4c4c',
    edgecolors='white',
    alpha=0.8
)

axs['shot'].set_title(
    "Mapa de tiros (tamaño = xG)",
    color='white',
    fontsize=15
)

# Títulos
plt.suptitle(
    "Análisis Individual – Lionel Messi | Mundial 2022",
    color='white',
    fontsize=22,
    y=1.05
)

plt.show()


<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

In [ ]:
scat_df = df_mundial.groupby(['player', 'team']).agg(
    tiros=('type', lambda x: (x == 'Shot').sum()),
    xg_total=('shot_statsbomb_xg', 'sum'),
    pases_int=('type', lambda x: (x == 'Pass').sum()),
    pases_comp=('type', lambda x: (
        (df_mundial.loc[x.index, 'type'] == 'Pass') &
        (df_mundial.loc[x.index, 'pass_outcome'].isna())
    ).sum()),
    intercepciones=('type', lambda x: (x == 'Interception').sum()),
    faltas=('type', lambda x: (x == 'Foul Committed').sum())
).reset_index()

scat_df['pct_pases'] = scat_df.pases_comp / scat_df.pases_int * 100

# Gráfico
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.style.use('dark_background')

fig = plt.figure(figsize=(20, 12))
gs = GridSpec(
    2, 3,
    width_ratios=[1, 0.75, 1],  # columna central más angosta
    height_ratios=[1, 1],
    wspace=0.15,
    hspace=0.25
)

fig.suptitle("Análisis Global de Jugadores – Mundial 2022", fontsize=24, y=0.97)

# Subplots
ax_xg      = fig.add_subplot(gs[0, 0])
ax_pass    = fig.add_subplot(gs[0, 2])
ax_def     = fig.add_subplot(gs[1, 0])
ax_mix     = fig.add_subplot(gs[1, 2])
ax_text    = fig.add_subplot(gs[:, 1])  # columna central completa

# Top 3 por categoría
top_attack = scat_df.sort_values('xg_total', ascending=False).head(3)
top_pass   = scat_df.sort_values('pases_comp', ascending=False).head(3)
top_def    = scat_df.sort_values('intercepciones', ascending=False).head(3)
top_mix    = scat_df.sort_values(['xg_total', 'pct_pases'], ascending=False).head(3)

def format_top(df):
    return "\n".join([f"{i+1}. {r.player}" for i, r in df.reset_index().iterrows()])

texto_central = (
    "TOP JUGADORES – MUNDIAL 2022\n\n"
    "ATAQUE (xG)\n"
    f"{format_top(top_attack)}\n\n"
    "CONTROL DE JUEGO (Pases)\n"
    f"{format_top(top_pass)}\n\n"
    "DEFENSA (Intercepciones)\n"
    f"{format_top(top_def)}\n\n"
    "JUGADORES COMPLETOS\n"
    f"{format_top(top_mix)}"
)

# xG vs tiros
ax_xg.scatter(scat_df.tiros, scat_df.xg_total, alpha=0.5)
for _, r in top_attack.iterrows():
    ax_xg.annotate(r.player, (r.tiros, r.xg_total),
                   xytext=(6, 6), textcoords='offset points', fontsize=9)

ax_xg.set_title("xG Total vs Tiros")
ax_xg.set_xlabel("Tiros Totales")
ax_xg.set_ylabel("xG Total")
ax_xg.grid(alpha=0.2)

# Pases
ax_pass.scatter(scat_df.pases_comp, scat_df.pct_pases, alpha=0.5)
for _, r in top_pass.iterrows():
    ax_pass.annotate(r.player, (r.pases_comp, r.pct_pases),
                     xytext=(6, 6), textcoords='offset points', fontsize=9)

ax_pass.set_title("Volumen y Precisión de Pase")
ax_pass.set_xlabel("Pases Completados")
ax_pass.set_ylabel("% Precisión de Pase")
ax_pass.grid(alpha=0.2)

# Defensa
ax_def.scatter(scat_df.intercepciones, scat_df.faltas, alpha=0.5)
for _, r in top_def.iterrows():
    ax_def.annotate(r.player, (r.intercepciones, r.faltas),
                    xytext=(6, 6), textcoords='offset points', fontsize=9)

ax_def.set_title("Intercepciones vs Faltas")
ax_def.set_xlabel("Intercepciones")
ax_def.set_ylabel("Faltas Cometidas")
ax_def.grid(alpha=0.2)

# Jugadores Completos
ax_mix.scatter(scat_df.pct_pases, scat_df.xg_total, alpha=0.5)
for _, r in top_mix.iterrows():
    ax_mix.annotate(r.player, (r.pct_pases, r.xg_total),
                    xytext=(6, 6), textcoords='offset points', fontsize=9)

ax_mix.set_title("Jugadores Completos")
ax_mix.set_xlabel("% Precisión de Pase")
ax_mix.set_ylabel("xG Total")
ax_mix.grid(alpha=0.2)

# Texto central
ax_text.axis('off')
ax_text.text(
    0.5, 0.5,
    texto_central,
    ha='center',
    va='center',
    fontsize=14,
    color='#f1c40f',
    bbox=dict(facecolor='black', alpha=0.8, boxstyle='round,pad=0.8')
)

plt.show()


<div style="height: 4px; background: linear-gradient(to right, #004d40, #4db6ac); border-radius: 2px; margin: 20px 0;"></div>

<a id='extras'></a>
<div style="padding: 20px; border-radius: 10px; border: 1px dashed #fbc02d;">
    <h2 style="color: #fbc02d; margin-top: 0;">Módulos Extras (Opcionales)</h2>
    <p style="margin-bottom: 0;"><b>Cálculo de Pases Progresivos:</b> Implementación de lógica espacial para detectar pases que ganan terreno.<br>
</div>

[Volver al índice](#indice)

In [ ]:
import numpy as np

# Filtrar pases
df_pases = df_mundial[
    (df_mundial.type == 'Pass') &
    (df_mundial.pass_outcome.isna()) &
    (df_mundial.location.notna()) &
    (df_mundial.pass_end_location.notna())
].copy()

# Coordenadas de inicio y fin
df_pases['x'] = df_pases['location'].apply(lambda x: x[0])
df_pases['y'] = df_pases['location'].apply(lambda x: x[1])

df_pases['end_x'] = df_pases['pass_end_location'].apply(lambda x: x[0])
df_pases['end_y'] = df_pases['pass_end_location'].apply(lambda x: x[1])

# Coordenadas del arco rival

# Distancia al arco rival
df_pases['dist_inicio'] = np.sqrt(
    (120 - df_pases['x'])**2 +
    (40 - df_pases['y'])**2
)

df_pases['dist_fin'] = np.sqrt(
    (120 - df_pases['end_x'])**2 +
    (40 - df_pases['end_y'])**2
)

# Pase progresivo
df_pases['es_progresivo'] = (
    (df_pases['dist_inicio'] - df_pases['dist_fin']) >
    (df_pases['dist_inicio'] * 0.25)
)

# Top 5 jugadores con más pases progresivos
top_prog = (
    df_pases[df_pases.es_progresivo]
    .groupby('player')
    .size()
    .sort_values(ascending=False)
    .head(5)
)

print("Top 5 Jugadores con más Pases Progresivos:")
print(top_prog)
